# 04 — The estimators & their parameters

*Module 02 · Naive Bayes — notebook 4 of 5*

You have built Naive Bayes by hand three times over — and in notebook 3 you saw that your by-hand
Gaussian version *is* scikit-learn's `GaussianNB`. Now we pick up the real estimator and learn its
**dials**: the few parameters that change how it behaves. We will turn each one and **watch** what it
does, rather than take anyone's word for it.

**Prerequisites:** notebooks 01–03 (Bayes by hand; the Gaussian likelihood; by-hand == `GaussianNB`);
module 00 notebooks 04 and 10 (the train/test split and cross-validation).

**What you will be able to do**

- Name the three members of the Naive Bayes family and when each fits.
- Turn `var_smoothing`, `alpha`, and `priors`, and predict the effect of each.
- Choose a parameter **honestly** — by cross-validation on the training data, spending the test set once.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import GaussianNB

from ml_course import datasets, viz
from ml_course.colors import CLASS_CYCLE, COLORS

np.random.seed(0)
viz.use_course_style()

df = datasets.load_penguins()
X, y = datasets.penguins_xy(df)
Xnp = X.to_numpy()   # fit on a plain array so later grid predictions raise no feature-name warning
recap = (GaussianNB().fit(Xnp, y).predict(Xnp) == y.to_numpy()).mean()
print(f"GaussianNB train accuracy (recap of notebook 3): {recap:.4f}")

## The Naive Bayes family

Every Naive Bayes uses the same Bayes' rule and the same naive product. They differ in **one choice**:
the shape of the likelihood P(feature | class).

- **`GaussianNB`** — a Gaussian per feature, for **continuous** measurements (notebook 3, and the
  penguins here).
- **`MultinomialNB`** — for **counts**, such as how many times each word appears in a document (the text
  case, notebook 5).
- **`BernoulliNB`** — for **presence or absence**, such as whether each word appears at all.

Pick the likelihood that matches your data; the rest is the Bayes machinery you already know. We tune
the Gaussian here, and meet the count models' main dial along the way.

## Dial 1 — `var_smoothing`

`GaussianNB` adds a tiny number to every feature's variance before using it — a **floor** that keeps a
variance from ever being zero. Why it matters at both ends:

- **too little** — a nearly constant feature has a near-zero variance, so its Gaussian becomes a tall
  thin spike, and a single outlier can dominate the whole prediction;
- **too much** — every feature's variance is inflated until all the classes look equally broad, and the
  feature stops telling them apart.

The default is a tiny floor (1e-9 of the largest variance). Let us sweep it and watch.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
vs_grid = [1e-9, 1e-6, 1e-3, 1e-2, 1e-1, 1.0, 10.0]
vs_scores = [cross_val_score(GaussianNB(var_smoothing=v), Xnp, y, cv=cv).mean() for v in vs_grid]
for v, s in zip(vs_grid, vs_scores, strict=True):
    print(f"var_smoothing={v:<8}: 5-fold CV accuracy {s:.4f}")

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.plot(vs_grid, vs_scores, color=COLORS["model"], marker="o", linewidth=2)
ax.set_xscale("log")
ax.set_xlabel("var_smoothing (log scale)")
ax.set_ylabel("5-fold CV accuracy")
ax.set_title("var_smoothing: harmless until it floods every variance")
plt.show()

**Read the figure.** From the default up to about 0.1 the accuracy does not budge — the floor sits
far below the real variances, so it changes nothing. Push it to 1.0 and accuracy dips (0.989); push it
to 10 and it falls to 0.711, as every variance is inflated until the classes look equally broad and the
spread stops telling them apart — accuracy sliding toward the majority-class baseline you met in module
00. The lesson: `var_smoothing` is a safety floor, not a knob you turn up to improve anything.

## Dial 2 — `alpha`, and the cure for the zero-frequency trap

Remember notebook 1's uncomfortable result: no Adélie in our data had a long bill, so P(long | Adélie)
came out **exactly zero** — which declared a long-billed Adélie *impossible* and would zero out the
entire product. The cure is **Laplace (add-α) smoothing**: add a small pseudo-count α to every bin's
count, so no category is ever exactly zero. Let us heal that very zero.

In [ ]:
# Notebook 1's bill_length contingency (Adelie has zero long-billed birds).
bins, labels = [30, 42, 48, 62], ["short", "medium", "long"]
binned = pd.cut(df["bill_length_mm"], bins=bins, labels=labels, include_lowest=True)
counts = pd.crosstab(df["species"], binned)
prior = df["species"].value_counts(normalize=True)

alphas = [0.0, 0.5, 1.0, 5.0]
p_long, post_long = [], []
for a in alphas:
    lik = (counts + a).div((counts + a).sum(axis=1), axis=0)   # Laplace-smoothed P(bin | species)
    post = lik.mul(prior, axis=0)
    post = post.div(post.sum(axis=0), axis=1)                  # P(species | bin)
    pl, pa = lik.loc["Adelie", "long"], post.loc["Adelie", "long"]
    p_long.append(pl)
    post_long.append(pa)
    print(f"alpha={a:<4}: P(long|Adelie)={pl:.4f}   posterior(Adelie|long)={pa:.4f}")

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.plot(alphas, p_long, color=CLASS_CYCLE[0], marker="o", linewidth=2, label="P(long | Adelie)")
ax.plot(alphas, post_long, color=COLORS["highlight"], marker="s", linewidth=2,
        label="posterior P(Adelie | long)")
ax.set_xlabel("alpha (Laplace pseudo-count)")
ax.set_ylabel("probability")
ax.set_title("alpha lifts notebook 1's impossible zero off the floor")
ax.legend()
plt.show()

**Read the figure.** At α = 0 we are back to notebook 1: P(long | Adélie) = 0 and the posterior
calls an Adélie there impossible. Any α > 0 lifts both off the floor — at α = 1, P(long | Adélie) rises
to a small 0.0065 and the posterior to a believable 0.018 (the posterior sits a little above the
likelihood because Adélie is the more common class, so Bayes weights it up). The data still says a
long-billed Adélie is *unlikely*, no longer *impossible*. This α is exactly the `alpha` of `MultinomialNB`, the count model we
turn loose on real word-counts in notebook 5; α → 0 is the log(0) we were careful to avoid in notebook
3.

## Dial 3 — `priors` (and `fit_prior`)

The **prior** is the belief you hold before seeing the measurement. By default Naive Bayes **learns** it
from the class frequencies in the training data (`fit_prior=True`). You can also **set** it by hand —
useful when you know the real base rate, or when one kind of mistake costs more than the other. Wherever
you place it, the prior tilts the decision.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
for ax, pg in zip(axes, [0.15, 0.85], strict=True):
    model = GaussianNB(priors=[1 - pg, pg]).fit(Xnp, y)
    viz.plot_decision_boundary(model, X, y, ax=ax)
    n_gentoo = int((model.predict(Xnp) == "Gentoo").sum())
    ax.set_title(f"P(Gentoo) = {pg}  ->  {n_gentoo} predicted Gentoo")
fig.tight_layout()
plt.show()

# A borderline penguin: how its prediction flips as the prior moves.
borderline = np.array([[40.8, 208.0]])
for pg in [0.15, 0.5, 0.85]:
    model = GaussianNB(priors=[1 - pg, pg]).fit(Xnp, y)
    pred = model.predict(borderline)[0]
    pgentoo = model.predict_proba(borderline)[0, 1]
    print(f"P(Gentoo)={pg}: borderline penguin -> {pred:7}  (P(Gentoo)={pgentoo:.3f})")

**Read the figure.** As the Gentoo prior rises from 0.15 to 0.85, the boundary slides down and to
the left — the model now needs less evidence to call a penguin Gentoo, so the amber region grows
(124 → 127 predicted Gentoo). Most penguins sit far from the boundary and never change, but the one
borderline bird (printed below the panels) flips from Adélie to Gentoo. On data this separable the prior
barely moves the score; it earns its keep when classes overlap, or when a false negative and a false
positive cost differently.

## Choosing a parameter honestly

How do you pick a value for a dial? With the discipline from module 00: try candidates by
**cross-validation on the training set**, keep the best, and touch the **test set once** at the very end
to estimate how the chosen model generalizes. Never select a setting by its test-set score — that spends
the test set on tuning and leaves you with an optimistic number.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    Xnp, y, test_size=0.3, stratify=y, random_state=0
)
search = GridSearchCV(
    GaussianNB(),
    {"var_smoothing": [1e-9, 1e-6, 1e-3, 1e-1, 1.0]},
    cv=cv,
)
search.fit(X_train, y_train)
best = search.best_params_["var_smoothing"]
print(f"CV-chosen var_smoothing: {best}  (CV accuracy {search.best_score_:.4f})")
print(f"sealed test accuracy (spent once): {search.score(X_test, y_test):.4f}")

**Read the output.** Cross-validation picks a `var_smoothing`, and we report a single sealed test
accuracy for it. On data this clean the default was already near the best, so the gain from tuning is
small — but the **workflow** is the lesson, not the gain. On the noisy, high-dimensional text of notebook
5, the same discipline is what separates an honest result from a fooled one.

## A limit we will measure later: calibration

Naive Bayes makes strong **decisions**, but its **probabilities** deserve a warning. Because it treats
correlated features as independent, it can *double-count* evidence and become **over-confident** —
reporting 0.999 where 0.85 would be honest. On these well-separated penguins it is actually
well-calibrated: its predicted probabilities, scored on held-out data, sit close to the truth. The
over-confidence is a high-dimensional, correlated-feature effect — so we save the honest measurement (a
reliability diagram and the **Brier score**, a standard calibration metric) for the text case in
notebook 5, where it genuinely shows.

## Your turn

1. **Find the cliff.** Sweep `var_smoothing` over a finer range and find roughly where the CV accuracy
   starts to fall. Why does flooding the variances hurt?
2. **Heal it yourself.** Recompute the Laplace-smoothed posterior P(Adélie | long) for α = 2 by hand from
   the contingency counts. Does it land between the α = 1 and α = 5 values you saw?
3. **(Harder)** Describe a situation where you would **set** a non-uniform prior rather than learn it from
   the data — for instance a known real-world base rate, or a setting where a missed positive costs far
   more than a false alarm.

## What you built

You took the Naive Bayes estimator off the shelf and learned to drive it. You met its **family**
(Gaussian, multinomial, Bernoulli — one per likelihood shape), turned **`var_smoothing`** until it
flooded the variances, used **`alpha`** to heal notebook 1's impossible zero, and watched **`priors`**
tilt the decision at the margin. Then you chose a parameter the honest way — cross-validate on train,
test once.

**Vocabulary**

- **`var_smoothing`** — a floor added to every feature's variance in `GaussianNB`; a safety net, not a
  tuning knob.
- **`alpha` (Laplace/Lidstone smoothing)** — a pseudo-count added to every category in the count models;
  the cure for the zero-frequency trap.
- **`priors` / `fit_prior`** — the class beliefs before the measurement: learned from frequencies, or set
  by hand under imbalance or asymmetric costs.
- **calibration** — whether predicted probabilities can be trusted as probabilities (measured in
  notebook 5).

## Going further (optional)

- **The count models.** `MultinomialNB` and `BernoulliNB` are the natural fit for text; notebook 5 builds
  the word-count representation by hand and turns `MultinomialNB` loose on it.
- **Discrete categories.** `CategoricalNB` handles features that are categories (not counts, not
  Gaussian) — the island a penguin lives on, say.
- **Trustworthy probabilities.** When you need calibrated probabilities, `CalibratedClassifierCV` wraps a
  classifier and recalibrates its scores. We meet calibration properly in notebook 5.

## References

- F. Pedregosa et al. (2011). *Scikit-learn: Machine Learning in Python.* Journal of Machine Learning
  Research, 12:2825–2830. The `sklearn.naive_bayes` estimators and their defaults (`var_smoothing`,
  `alpha`, `fit_prior`).
- G. James, D. Witten, T. Hastie, R. Tibshirani (2021). *An Introduction to Statistical Learning*
  (§4.4.4). DOI: [10.1007/978-1-0716-1418-1](https://doi.org/10.1007/978-1-0716-1418-1).

---

*Previous: 03 — The Gaussian likelihood* · *Next: 05 — Text classification (the demanding case)*